# 19 · CP8 Claims-Edit generation (T4; Kolors-Inpainting)

**T4 runtime, fp16, two-stage load.** Runtime → Change runtime type → T4 GPU (a standard T4 with 12.7 GB of system RAM is enough). CPU offload needed about 18 GB of RAM and crashed a standard T4 (28 Nov), so the text encoder (ChatGLM-6B) is loaded alone onto the GPU, encodes the plan's prompts and is freed; then the UNet and VAE are loaded onto the GPU. Data generation only: the detector never runs here (its scoring is the `cp8` block of notebook 17 on an L4), so the T4 parity result does not apply.

**Model** (Dhanush, 27 Nov): Kolors-Inpainting, Hugging Face `Kwai-Kolors/Kolors-Inpainting` @ `3bb98b424c9a`, Apache-2.0, through the Kolors repository's pipeline (`Kwai-Kolors/Kolors` @ `038818d244ed`, installed with `--no-deps`). Kolors is a P2 held-out editor (never trained on); it is not a training editor (asserted below). CP8 is therefore not independent of the P2 Kolors pixels.

**What it does:**
1. Reads the 150 reserved COCO val2017 photos (the verified `claims_edit_src` tar) and their instance annotations.
2. Plans one edit per photo (mask: the largest instance of car, couch, chair, dining table or laptop with 2–40% area; damage and prompt by a seeded hash; seed per photo) and **freezes the plan by SHA-256** in `eval_subsets.json` (key `CP8_plan`) before any image is generated; a changed plan is refused.
3. **`SMOKE_ONLY = True` (default):** 2 images only; records s/image, peak GPU memory and output size, estimates CU for 150, and **stops**. If Kolors fails (out of memory or more than about 2 min per image), it stops and asks; there is no silent model switch.
4. **`SMOKE_ONLY = False`** (only on Dhanush's word): all photos, resumable; the edit is composited inside the mask dilated by 8 iterations (cross structuring element) (a hard seam, stated in the spec card); a pair is dropped if generation fails or the edit is not visible (mean change < 10/255); more than 10% dropped fails CP8. Writes the pairs and the `claims_edit` tar (the real photos and their edited copies with masks) for notebook 17.

**Results** (send to Claude): `Paper 1/runs/cp8/cp8_generation_seed42.json` and `compute_cp8_generation.json`, with the balance.

In [ ]:
# Cell 1: mount Drive, get the code, install it
import os, subprocess, sys

DRY = os.environ.get('EVIGROUND_DRY_RUN') == '1'   # CI / local test: fake sources, temp folders
REPO_URL = 'github.com/DHANUSHKUMMARI/Paper1.git'
BRANCH = 'main'                                    # 'claude/nice-curie-3tzxzr' before the merge

if DRY:
    REPO = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
else:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    REPO = '/content/Paper1'
    if not os.path.exists(REPO):
        try:
            tok = userdata.get('GITHUB_TOKEN')       # never printed
        except Exception:
            tok = None
        url = f'https://{tok}@{REPO_URL}' if tok else f'https://{REPO_URL}'
        r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, REPO],
                           capture_output=True, text=True)
        if r.returncode != 0:                        # stderr is not shown: it can contain the URL
            raise RuntimeError('git clone failed: check GITHUB_TOKEN (Secrets panel) and BRANCH')
        subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'],
                       check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO], check=True)
    # Colab ships torchao 0.10; peft refuses a torchao older than 0.16 with an ImportError when it
    # adds the LoRA layers, although EviGround never uses torchao: remove an old one.
    import importlib, importlib.metadata as _md
    from packaging.version import Version
    try:
        if Version(_md.version('torchao')) < Version('0.16.0'):
            subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=True)
            importlib.invalidate_caches()
    except _md.PackageNotFoundError:
        pass

sys.path.insert(0, os.path.join(REPO, 'src'))
# Re-running in a runtime that already imported eviground would keep the OLD code in memory
# although git pulled the new one: drop it so the next import reads the files on disk.
for _name in [m for m in sys.modules if m == 'eviground' or m.startswith('eviground.')]:
    del sys.modules[_name]
print('repo:', REPO, '| branch:', BRANCH, '| dry run:', DRY)

# Kolors-Inpainting runs through the Kolors repository's own pipeline (its README), pinned, installed
# with --no-deps. Its requirements pin transformers 4.42.4, whose tokenizers 0.19 has no Python 3.13
# wheel and does not compile for 3.13 (28 Nov): transformers 4.45.2 + tokenizers 0.20.3 instead (the
# oldest pair that installs here), with inpaint.patch_chatglm_pad; token ids checked identical.
# Installed HERE, before anything imports torch: if torch changed, restart and Run all again.
KOLORS_PINS = ['diffusers==0.28.2', 'transformers==4.45.2', 'tokenizers==0.20.3', 'huggingface-hub==0.23.4',
               'accelerate==0.27.2', 'sentencepiece']
KOLORS_REPO_SPEC = 'git+https://github.com/Kwai-Kolors/Kolors@038818d244ed103056abd10f429729a26af4d239'
TORCH_BEFORE = TORCH_AFTER = None
if not DRY:
    import importlib, importlib.metadata as _md
    def _version(pkg):
        try:
            return _md.version(pkg)
        except _md.PackageNotFoundError:
            return None
    def _run(cmd, what):
        # output captured, its last lines shown on failure (a bare -q hid the cause on 28 Nov)
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            print('\n'.join((r.stdout + r.stderr).strip().splitlines()[-30:]))
            raise RuntimeError(f'{what} failed (exit {r.returncode}); its last 30 lines are above')
    TORCH_BEFORE = _version('torch')
    # huggingface-hub too: eviground needs >= 0.24, so `pip install -e REPO` above upgrades it on every
    # Run all, and diffusers 0.28.2 needs cached_download (removed in hub 0.26); 28 Nov
    if (_version('kolors') is None or _version('diffusers') != '0.28.2' or _version('transformers') != '4.45.2'
            or _version('huggingface-hub') != '0.23.4'):
        _run([sys.executable, '-m', 'pip', 'install', *KOLORS_PINS], 'the Kolors library pins')
        _run([sys.executable, '-m', 'pip', 'install', '--no-deps', KOLORS_REPO_SPEC], 'the Kolors repository')
        importlib.invalidate_caches()
        if 'huggingface_hub' in sys.modules or 'diffusers' in sys.modules:   # a cached import keeps the old hub
            print('pins reinstalled after an import: Runtime -> Restart session, then Run all')
            raise RuntimeError('the Kolors pins were reinstalled in a session that had already imported them: Runtime -> Restart session, then Run all')
    # Colab ships peft 0.21, which needs accelerate >= 1.x (clear_device_cache); Kolors pins accelerate
    # 0.27.2, and diffusers 0.28.2 imports peft whenever it is installed (register_modules). Nothing
    # in this notebook uses peft, so remove it (28 Nov). diffusers caches the check at import, so a
    # session that already imported diffusers must restart.
    if _version('peft') is not None:
        _run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'peft'], 'removing peft')
        importlib.invalidate_caches()
        if 'diffusers' in sys.modules:
            print('peft removed: Runtime -> Restart session, then Run all')
            raise RuntimeError('peft was removed after diffusers was imported: Runtime -> Restart session, then Run all')
    TORCH_AFTER = _version('torch')
    print('torch', TORCH_BEFORE, '->', TORCH_AFTER, '| diffusers', _version('diffusers'), '| kolors', _version('kolors'))
    if TORCH_AFTER != TORCH_BEFORE:
        print('Runtime -> Restart session, then Run all')
        raise RuntimeError('the install replaced torch: Runtime -> Restart session, then Run all (the install is skipped then)')

In [ ]:
# Cell 2: settings
SMOKE_ONLY = True          # True: 2 images, then stop and report; False: all photos (Dhanush's word only)
CU_BALANCE_START = None    # REQUIRED on a paid runtime: the balance Colab shows now
SEED = 42
N_SMOKE = 2
MAX_S_PER_IMAGE = 120      # more than this per image on the T4: stop and ask (Dhanush, 27 Nov)
MAX_FAILS = 3              # this many generation failures in a row: stop and report (review of PR #65)
MIN_RAM_GB = 12           # two-stage load (28 Nov): a standard T4 (12.7 GB) is enough
import datetime, json, os, shutil, tempfile, time, zipfile
from pathlib import Path
import numpy as np
from PIL import Image
from eviground.data import claims_edit as ce, inpaint, testcache
from eviground.data.cp1 import coco_instances
from eviground.data.registry import REGISTRY
from eviground.env import get_paths, session_info
from eviground.utils import session
from eviground.eval import phasec

T0 = time.time()
info = session_info(cu_per_hour=session.CU_RATES['T4'])
paths = get_paths()
if DRY:
    from eviground.env import Paths
    _dry = Path(tempfile.gettempdir()) / 'eviground_cp8_dry'
    shutil.rmtree(_dry, ignore_errors=True)
    paths = Paths(root=_dry / 'drive' / 'Paper 1', local_data=_dry / 'data')
    SMOKE_ONLY = False         # the dry run does the smoke test and the full run with the mock
paths.ensure()
CK = paths.runs / 'checkpoints'
SUBSETS_FILE = CK / 'eval_subsets.json'
CACHE = paths.runs / 'cache' / 'tests'
OUT = paths.runs / 'cp8'
PAIRS = OUT / 'pairs'
LOCAL = paths.local_data / 'cp8'
RESULT = OUT / f'cp8_generation_seed{SEED}.json'
for d in (OUT, PAIRS, LOCAL):
    d.mkdir(parents=True, exist_ok=True)
COCO_ZIP = paths.datasets / REGISTRY['coco_val2017'].folder / 'annotations_trainval2017.zip'
WEIGHTS = Path('/content/kolors_weights') if not DRY else None
GPU = info.get('gpu_name')
print('commit', info['git_commit'], '| gpu', GPU, '| smoke only', SMOKE_ONLY)

In [ ]:
# Cell 3: in the dry run a fake COCO world (photos, polygons, the claims_edit_src tar); then the pre-flight
if DRY:
    rng = np.random.default_rng(0)
    photos, images, anns = [], [], []
    src = paths.local_data / 'fake_coco'
    src.mkdir(parents=True, exist_ok=True)
    for i, iid in enumerate((139, 285, 632, 724, 776)):
        w, h = [(128, 96), (96, 128), (120, 90), (128, 80), (100, 100)][i]
        p = src / f'{iid:012d}.jpg'
        Image.fromarray((rng.random((h, w, 3)) * 255).astype(np.uint8)).save(p, quality=92)
        photos.append({'uid': f'coco/val2017/{iid:012d}.jpg', 'path': str(p), 'source': 'coco', 'label': 'real', 'role': 'test_P7'})
        images.append({'id': iid, 'width': w, 'height': h, 'file_name': p.name})
        s = int(min(w, h) * 0.5)   # about 20-30% of the image
        anns.append({'id': len(anns) + 1, 'image_id': iid, 'category_id': [3, 62, 73, 63, 67][i], 'area': s * s,
                     'iscrowd': 0, 'segmentation': [[5, 5, 5 + s, 5, 5 + s, 5 + s, 5, 5 + s]], 'bbox': [5, 5, s, s]})
    cats = [{'id': 3, 'name': 'car'}, {'id': 62, 'name': 'chair'}, {'id': 63, 'name': 'couch'},
            {'id': 67, 'name': 'dining table'}, {'id': 73, 'name': 'laptop'}]
    COCO_ZIP.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(COCO_ZIP, 'w') as zf:
        zf.writestr('annotations/instances_val2017.json', json.dumps({'images': images, 'annotations': anns, 'categories': cats}))
    _build = paths.local_data / 'build'
    testcache.publish(testcache.build(photos, 'claims_edit_src', _build), _build, CACHE, seed=SEED)
if not DRY:
    assert session.gpu_type(GPU) == 'T4', f'CP8 generation runs on a T4 (fp16), found {GPU}'
    assert isinstance(CU_BALANCE_START, (int, float)), 'type the Colab balance into CU_BALANCE_START (cell 2)'
    RAM_GB = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9
    print(f'system RAM {RAM_GB:.1f} GB')
    if RAM_GB < MIN_RAM_GB:
        print(f'WARNING: {RAM_GB:.1f} GB < {MIN_RAM_GB} GB: Kolors may run out of memory while loading')
EST_CU = 0.3 if SMOKE_ONLY else 3.0   # estimates; the smoke test measures the real rate
if DRY:
    EST_CU = 0.0
MODEL_LOG = inpaint.assert_not_training_editor(inpaint.KolorsInpainter.name)   # CP8 pass rule, logged
print(MODEL_LOG)
PREFLIGHT = session.preflight(
    {'notebook': '19 CP8 generation', 'gpu': GPU, 'model': inpaint.KolorsInpainter.name, 'smoke only': SMOKE_ONLY,
     'result file': RESULT},
    paths={'claims_edit_src tar': testcache.paths(CACHE, 'claims_edit_src')[0], 'COCO annotations zip': COCO_ZIP},
    est_cu=EST_CU, cu_left=CU_BALANCE_START)

In [ ]:
# Cell 4: the plan, frozen by SHA-256 in eval_subsets.json (key CP8_plan) before any generation
ROWS = testcache.stage(CACHE, 'claims_edit_src', LOCAL / 'src')
with zipfile.ZipFile(COCO_ZIP) as zf:
    INSTANCES = json.loads(zf.read('annotations/instances_val2017.json'))
PLAN = ce.freeze(ce.plan_edits(ROWS, INSTANCES, seed=SEED))
STORED = json.loads(SUBSETS_FILE.read_text()) if SUBSETS_FILE.is_file() else {}
if 'CP8_plan' in STORED:
    assert ce.intact(STORED['CP8_plan']), 'CP8_plan in eval_subsets.json was changed: stop and ask Dhanush'
    assert STORED['CP8_plan']['sha256'] == PLAN['sha256'], 'the frozen CP8 plan differs from a fresh plan: stop and ask'
    print('CP8_plan already frozen')
else:
    STORED['CP8_plan'] = PLAN
    SUBSETS_FILE.parent.mkdir(parents=True, exist_ok=True)
    SUBSETS_FILE.write_text(json.dumps(STORED, indent=1))
    print('CP8_plan frozen in', SUBSETS_FILE)
EYE_CHECK = ce.eye_check_sample(PLAN, 30, seed=SEED)
(OUT / 'eye_check_uids.json').write_text(json.dumps(EYE_CHECK, indent=1))
print(f"plan: {PLAN['images']} edits, {PLAN['dropped_count']} photos without a 2-40% instance | sha256 {PLAN['sha256']}")
BY_UID = {r['uid']: r for r in ROWS}

In [ ]:
# Cell 5: the inpainter (Kolors-Inpainting on the T4; the mock in the dry run)
if DRY:
    INP = inpaint.MockInpainter()
else:
    from huggingface_hub import snapshot_download
    snapshot_download(inpaint.KOLORS_WEIGHTS, revision=inpaint.KOLORS_WEIGHTS_REV, local_dir=WEIGHTS)
    # the text encoder runs once, at load, for every distinct prompt of the frozen plan
    INP = inpaint.KolorsInpainter(WEIGHTS, prompts=[p['prompt'] for p in PLAN['plans']], negative=ce.NEGATIVE)
print('inpainter:', INP.name)

In [ ]:
# Cell 6: generate (2 images in the smoke test; all photos otherwise). Resumes from pairs already on Drive.
def pair_paths(plan_row):
    stem = Path(plan_row['uid']).stem
    return PAIRS / f'{stem}_edited.png', PAIRS / f'{stem}_mask.png'

def run_one(p):
    orig = np.asarray(Image.open(BY_UID[p['uid']]['path']).convert('RGB'))
    ann = next(a for a in INSTANCES['annotations'] if a['id'] == p['ann_id'])
    mask = ce.dilate(ce.instance_mask(ann, p['width'], p['height']))
    t = time.time()
    try:
        gen = INP.inpaint(orig, mask, p['prompt'], negative=ce.NEGATIVE, seed=p['gen_seed'],
                          steps=ce.STEPS, guidance=ce.GUIDANCE)
        edited = ce.composite(orig, gen, mask)
    except Exception as e:   # noqa: BLE001 - a failed pair is counted by the drop rule
        print('  failed', p['uid'], type(e).__name__, e)
        edited = None
        error = f'{type(e).__name__}: {e}'[:300]
    else:
        error = None
    seconds = time.time() - t
    kept, reason = ce.keep_pair(orig, edited, mask)
    if error:
        rec_error = {'error': error}
    else:
        rec_error = {}
    rec = {'uid': p['uid'], 'kept': kept, 'reason': reason, 'seconds': round(seconds, 2), 'fingerprint': GEN_FP,
           'mean_change': None if edited is None else round(ce.visible_change(orig, edited, mask), 3), **rec_error}
    if kept:
        e_path, m_path = pair_paths(p)
        Image.fromarray(edited).save(e_path)
        Image.fromarray(mask.astype(np.uint8) * 255).save(m_path)
        rec['bytes'] = e_path.stat().st_size + m_path.stat().st_size
    return rec

RECORDS_FILE = OUT / 'pairs_records.jsonl'
GEN_FP = ce.generation_fingerprint(INP.name, PLAN['sha256'], phasec.code_tree(REPO, ce.GEN_CODE_PATHS))
print('generation fingerprint', GEN_FP[:12])
DONE = {}
if RECORDS_FILE.is_file():
    DONE = {r['uid']: r for r in (json.loads(l) for l in RECORDS_FILE.read_text().splitlines() if l.strip())}
    ce.check_resume(DONE.values(), GEN_FP)   # stops rather than mix generations in one tar
targets = PLAN['plans'][:N_SMOKE] if SMOKE_ONLY else PLAN['plans']
todo = ce.to_generate(targets, DONE)   # new pairs, and pairs whose generation failed (same seed)
print(f'{len(targets)} targets, {len(todo)} to generate')
STOPPED = None
with open(RECORDS_FILE, 'a') as fh:
    for i, p in enumerate(todo):
        rec = run_one(p)
        DONE[p['uid']] = rec
        fh.write(json.dumps(rec) + '\n'); fh.flush()
        print(f"  {i + 1}/{len(todo)} {p['uid']} {rec['reason']} {rec['seconds']} s")
        if rec['seconds'] > MAX_S_PER_IMAGE and not DRY:
            STOPPED = f"{rec['seconds']} s for one image > {MAX_S_PER_IMAGE} s: stop and ask Dhanush (no silent model switch)"
            print('STOP:', STOPPED)
            break
        if ce.consecutive_failures([DONE[q['uid']] for q in todo[:i + 1]]) >= MAX_FAILS:
            STOPPED = f"{MAX_FAILS} generation failures in a row (last: {rec.get('error')}): stop and report"
            print('STOP:', STOPPED)
            break

In [ ]:
# Cell 7: the smoke-test measurements and the CU estimate for 150 pairs; in the full run, the drops and the tar
done = [DONE[p['uid']] for p in targets if p['uid'] in DONE]
secs = [r['seconds'] for r in done if r['reason'] != 'generation failed']
s_per_image = float(np.median(secs)) if secs else None
peak = INP.peak_gpu_gb() if hasattr(INP, 'peak_gpu_gb') else None
sizes = [r['bytes'] for r in done if r.get('bytes')]
SMOKE = {'images': len(done), 's_per_image_median': s_per_image, 'peak_gpu_gb': peak,
         'output_bytes_per_pair_median': int(np.median(sizes)) if sizes else None,
         'failed': sum(r['reason'] == 'generation failed' for r in done)}
if s_per_image is not None:
    hours_150 = 150 * s_per_image / 3600
    SMOKE['cu_estimate_150'] = round(hours_150 * session.CU_RATES['T4'] + 0.15, 2)   # + about 0.15 CU set-up (estimate)
    SMOKE['cu_estimate_100'] = round(100 * s_per_image / 3600 * session.CU_RATES['T4'] + 0.15, 2)
print('smoke test:', SMOKE)
FULL = None
if not SMOKE_ONLY and STOPPED is None:
    kept = [p for p in PLAN['plans'] if DONE.get(p['uid'], {}).get('kept')]
    dropped_pairs = [{'uid': u, 'reason': r['reason']} for u, r in DONE.items() if not r['kept']]
    n_total = len(PLAN['plans']) + PLAN['dropped_count']
    drop_share = (len(dropped_pairs) + PLAN['dropped_count']) / n_total if n_total else None
    rows = []
    for p in PLAN['plans']:   # the real photo and, if kept, its edited copy with the mask
        r = BY_UID[p['uid']]
        rows.append({'uid': p['uid'], 'path': r['path'], 'source': 'coco', 'label': 'real',
                     'category': p['category'], 'pair': p['uid']})
        if DONE.get(p['uid'], {}).get('kept'):
            e_path, m_path = pair_paths(p)
            rows.append({'uid': ce.edited_uid(p['uid']), 'path': str(e_path), 'mask_path': str(m_path),
                         'source': 'claims_edit', 'label': 'edited', 'category': p['category'],
                         'damage': p['damage'], 'pair': p['uid']})
    build = LOCAL / 'build'
    MAN = testcache.publish(testcache.build(rows, 'claims_edit', build), build, CACHE, seed=SEED)
    FULL = {'kept_pairs': len(kept), 'dropped_pairs': dropped_pairs, 'photos_without_instance': PLAN['dropped_count'],
            'drop_share': drop_share, 'drop_rule_pass': drop_share is not None and drop_share <= 0.10,
            'tar': {'images': MAN['images'], 'masks': MAN['masks'], 'tar_sha256': MAN['tar_sha256']}}
    print('full run:', {k: v for k, v in FULL.items() if k != 'dropped_pairs'})

In [ ]:
# Cell 8: result file, compute record, release
RESULT_DATA = {
    'step': 'CP8 Claims-Edit generation', 'dry_run': DRY, 'branch': BRANCH, 'seed': SEED, 'session': info,
    'model': INP.name, 'model_check': MODEL_LOG, 'pins': {'weights': inpaint.KOLORS_WEIGHTS_REV, 'repo': inpaint.KOLORS_REPO_REV},
    'torch': {'before': TORCH_BEFORE, 'after': TORCH_AFTER}, 'plan_sha256': PLAN['sha256'], 'generation_fingerprint': GEN_FP, 'plan_images': PLAN['images'],
    'settings': ce.SETTINGS, 'smoke_only': SMOKE_ONLY, 'smoke': SMOKE, 'stopped': STOPPED, 'full': FULL,
    'eye_check_uids': EYE_CHECK, 'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
}
RESULT.write_text(json.dumps(RESULT_DATA, indent=1, default=str))
print('saved', RESULT)
COMPUTE = session.finish(OUT, T0, gpu=GPU or 'cpu', name='compute_cp8_generation.json', balance_start=CU_BALANCE_START,
                         extra={'git_commit': info['git_commit'], 'est_cu': EST_CU, 'smoke_only': SMOKE_ONLY})
if DRY:
    assert PLAN['images'] == 5 and ce.intact(json.loads(SUBSETS_FILE.read_text())['CP8_plan'])
    assert FULL['kept_pairs'] == 5 and FULL['drop_rule_pass'] and FULL['tar']['masks'] == 5
    assert FULL['tar']['images'] == 10 and testcache.is_ready(CACHE, 'claims_edit')
    assert SMOKE['images'] == 5 and SMOKE['cu_estimate_150'] is not None
    assert INP.calls == 5
RELEASED = session.release(DRY)
if DRY:
    assert RELEASED == 'dry', RELEASED